In [1]:
#This notebook is for trying out and compiling what will become a .py file for end-to-end, lightweight parsing of articles

In [11]:
import pandas
import nltk
from transformers import pipeline
import spacy
from fastcoref.modeling import FCoref, FCorefModel
import torch
import re
import sys
from pathlib import Path
import joblib

In [13]:
#Import trained models and modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT / 'create_model') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "create-model"))

from compute_features import compute_handcrafted_features

si = joblib.load(PROJECT_ROOT / "models" / "distilled" / "si.joblib")
tc = joblib.load(PROJECT_ROOT / "models" / "distilled" / "tc.joblib")

In [14]:
nltk.download('punkt')
nltk.download('punkt_tab')
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [15]:
if not hasattr(FCorefModel, "all_tied_weights_keys"):
    FCorefModel.all_tied_weights_keys = {}

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
coref_model = FCoref(device=device)

04/23/2026 13:34:45 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/23/2026 13:34:45 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/23/2026 13:34:45 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/23/2026 13:34:45 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/23/2026 13:34:45 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/23/2026 13:34:45 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/tokenizer_config.json

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
04/23/2026 13:34:46 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/main "HTTP/1.1 200 OK"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/mo

In [16]:
claim_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=-1
)

04/23/2026 13:34:46 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/valhalla/distilbart-mnli-12-3/ef9a58ce6a9cd44cd0d4c2f7db1cd67f81019a8b/config.json "HTTP/1.1 200 OK"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3 "HTTP/1.1 200 OK"
04/23/2026 13:34:46 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/commits/main "HTTP/1.1 200 OK"
04/23/2026 13:34:47 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/discussions?p=0 "HTTP/1.1 200 OK"
04/23/2026 13:34:47 - INFO - 	 HTTP Request: GET https://h

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

04/23/2026 13:34:47 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04/23/2026 13:34:47 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


In [17]:
def resolve_coreferences(text):
    """
    Replaces pronouns and implicit references in the text with their explicit entities.
    """
    #Predict coreference clusters
    preds = coref_model.predict(texts=[text])

    #Get clusters as character start/end indices
    clusters = preds[0].get_clusters(as_strings=False)

    replacements = []
    for cluster in clusters:
        #The first mention in a cluster is usually the explicit entity (the antecedent)
        primary_start, primary_end = cluster[0]
        primary_text = text[primary_start:primary_end]

        #Replace all subsequent mentions (usually pronouns) with the primary text
        for mention_start, mention_end in cluster[1:]:
            replacements.append((mention_start, mention_end, primary_text))

    #Sort replacements in reverse order of their start index
    replacements.sort(key=lambda x: x[0], reverse=True)

    #Apply replacements
    resolved_text = text
    for start, end, rep_text in replacements:
        resolved_text = resolved_text[:start] + rep_text + resolved_text[end:]

    return resolved_text

In [18]:
def format_claim(text):
    """Clean up capitalization and punctuation without ruining proper nouns."""
    if not text:
        return ""

    #1. Strip trailing/leading whitespace and weird trailing punctuation
    text = text.strip()
    text = re.sub(r'[,\s\.]+$', '', text)

    #2. Fix spaces before periods or commas
    text = re.sub(r'\s+([.,])', r'\1', text)

    #3. Handle double punctuation (e.g., ",." -> ".")
    text = re.sub(r',[.]', '.', text)

    #4. Final capitalization: Only touch the first character
    if len(text) > 0:
        text = text[0].upper() + text[1:]

    #5. Ensure it ends with exactly one period
    if not text.endswith('.'):
        text += '.'

    return text

In [19]:
def decompose_sentence(sentence):
    doc = nlp(sentence)

    #1. Identify potential split points (Verbs that start new clauses)
    potential_splits = []
    for token in doc:
        #A split is valid only if the token is a verb AND has its own subject
        if token.pos_ in ["VERB", "AUX"]:
            has_subj = any(t.dep_ in ["nsubj", "nsubjpass", "expl"] for t in token.children)

            #If it's a 'conj' or 'advcl' WITH its own subject, it's a new atomic claim
            if has_subj and token.dep_ in ["conj", "advcl"]:
                potential_splits.append(token)

    if not potential_splits:
        return [sentence.strip()]

    #2. Perform the slices
    split_indices = sorted([t.left_edge.i for t in potential_splits])

    claims = []
    last_idx = 0
    for idx in split_indices:
        #Grab the text from the last split to this one
        slice_end = idx
        if idx > 0 and doc[idx-1].pos_ in ["CCONJ", "SCONJ", "PUNCT"]:
            slice_end = idx - 1

        chunk = doc[last_idx:slice_end].text.strip()
        if len(chunk.split()) > 2: # Ignore tiny fragments
            claims.append(chunk)
        last_idx = idx

    #Add the final piece
    final_chunk = doc[last_idx:].text.strip()
    if final_chunk:
        #Clean up leading conjunctions like "and" or "as"
        temp_doc = nlp(final_chunk)
        if temp_doc[0].pos_ in ["CCONJ", "SCONJ"]:
            final_chunk = temp_doc[1:].text.strip()
        claims.append(final_chunk)

    return [format_claim(c) for c in claims]

In [20]:
def is_subjective(sentence):
    """Full text needs to be split into """
    result = claim_classifier(
            sentence,
            candidate_labels=["factual claim", "personal opinion"],
            multi_label=False
        )

    if result['labels'][0] == "factual claim":
        return False, result['scores'][0]
    else:
        return True, result['scores'][0]

In [21]:
def split_sentences(text: str):
    """Split text into sentences, returning (sentence_str, start_char, end_char)."""

    tokenizer = nltk.tokenize.punkt.PunktSentenceTokenizer()
    sents = []

    # .span_tokenize gives you (start, end) tuples
    for start, end in tokenizer.span_tokenize(text):
        s = text[start:end].strip()
        if s:
            # Clean newlines just like before
            s_clean = " ".join(s.split())
            sents.append((s_clean, start, end))

    return sents

In [ ]:
#How to load the models and run inference on a new article
si = joblib.load(SI_PATH)
tc = joblib.load(TC_PATH)


def predict_propaganda(article_text: str,
                        si_bundle: dict,
                        tc_bundle: dict) -> list[dict]:
    """Run the full propaganda detection pipeline on a raw article.

    Steps
    -----
    1. Split article into sentences.
    2. Score each sentence with the Span Identifier.
    3. For sentences above the detection threshold, classify the technique.

    Returns
    -------
    List of dicts, each with keys:
        'sentence'  : str   — the flagged sentence
        'span'      : [int, int]  — [start_char, end_char] in the article
        'technique' : str   — predicted propaganda technique
        'si_score'  : float — span identifier confidence (0–1)
    """
    sents = split_sentences(article_text)
    if not sents:
        return []

    sent_strings = [s for s, _, _ in sents]

    #Identify spans
    si_hc = si_bundle['feature_extractor'](sent_strings)
    si_feats = hstack([
        si_bundle['word_vectorizer'].transform(sent_strings),
        si_bundle['char_vectorizer'].transform(sent_strings),
        si_hc,
    ])
    si_scores  = si_bundle['classifier'].predict_proba(si_feats)[:, 1]
    threshold  = si_bundle['threshold']
    is_prop    = si_scores >= threshold

    flagged = [
        (sent_strings[i], sents[i][1], sents[i][2], float(si_scores[i]))
        for i in range(len(sents))
        if is_prop[i]
    ]

    if not flagged:
        return []

    #Classify techniques
    flagged_texts = [f[0] for f in flagged]
    tc_hc = tc_bundle['feature_extractor'](flagged_texts)
    tc_feats = hstack([
        tc_bundle['word_vectorizer'].transform(flagged_texts),
        tc_bundle['char_vectorizer'].transform(flagged_texts),
        tc_hc,
    ])

    #Get probabilities for all 14 classes
    tc_probs = tc_bundle['classifier'].predict_proba(tc_feats)
    class_names = tc_bundle['classifier'].classes_

    results = []
    for i, (sent, ss, se, si_score) in enumerate(flagged):
        sentence_probs = tc_probs[i]

        #Filter classes that meet the threshold
        hits = [
            (class_names[j], sentence_probs[j])
            for j in range(len(class_names))
            if sentence_probs[j] >= 0.2
        ]

        #If nothing hits threshold, take the top 1
        if not hits:
            top_idx = sentence_probs.argmax()
            hits = [(class_names[top_idx], sentence_probs[top_idx])]

        for tech_name, tech_score in hits:
            results.append({
                'sentence': sent,
                'span': [ss, se],
                'technique': tech_name,
                'tc_score': round(float(tech_score), 3),
                'si_score': round(si_score, 3),
            })

    return results

In [ ]:
def lightweight_pipeline(full_text):
    # Stage 1: Disambiguation (Coreference Resolution)—we're doing first instead of third because it works best with our #lightweight, API-free approach
    disambiguated_text = resolve_coreferences(full_text)

    # Stage 2: Split sentences
    complex_sentences = nltk.sent_tokenize(disambiguated_text)
    #prop_sentences = split_sentences(full_text)

    # Stage 3: Decomposition
    atomic_sentences = []
    for sent in complex_sentences:
        # Only try to decompose longer sentences with conjunctions
        if " and " in sent.lower() or " but " in sent.lower() or "," in sent:
            decomposed = decompose_sentence(sent)
            atomic_sentences.extend(decomposed)
        else:
            atomic_sentences.append(sent)

    # Stage 4: Identify potential propaganda
    spans =  predict_propaganda(full_text, si, tc)

    #Stage 5: Evaluate whether each is a factual claim or an opinion
    claims = []
    for claim in atomic_sentences:
        if len(claim.split()) < 4:
            continue
        result = is_subjective(claim)
        if result[0] == False and result[1] > 0.6:
            claims.append(format_claim(claim))

    return {'propaganda_spans': spans, 'factual_claims': list(set(claims))}


In [ ]:
lightweight_pipeline("The partnership between John and Jane illustrates the importance of collaboration. They are very very good together, and it's no surprise as they are American born and bred. The GDP is rising in this country, and you can see why in this beautiful heterosexual partnership.")